In [0]:
dbutils.widgets.text("current_date", "")
current_date = dbutils.widgets.get("current_date")
print(current_date)
# Configuration
source_dir = f"/Volumes/zoomcarprocessingpipeline/default/processing_data/customers/source/zoom_car_customers_{current_date}.json"
archive_dir = f"/Volumes/zoomcarprocessingpipeline/default/processing_data/customers/archive/zoom_car_customers_{current_date}.json"
stage_table = "`zoomcarprocessingpipeline`.default.customers_stage"
error_table = "`zoomcarprocessingpipeline`.default.customers_errors"

print(f"Processing orders data from: {source_dir}")
print(f"Staging table: {stage_table}")

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json


# Define schema for booking data
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("status", StringType(), True)
])

print("Schema defined for orders data")
rawcustomers_df = (spark.read.schema(customer_schema).option("multiline", "true").json(source_dir))

rawcustomers_df.show()

In [0]:
# Check for null values in each column
null_counts = rawcustomers_df.agg(
    *[F.sum(F.col(column).isNull().cast("int")).alias(f"{column}_null_count") for column in rawcustomers_df.columns]
)

# Check for data types
data_type_checks = [F.col(column).cast("string").alias(f"{column}_type_check") for column in rawcustomers_df.columns]

# Apply the data type checks
df_check = rawcustomers_df.select(data_type_checks)

# Show the results of the checks
print("Null Counts:")
null_counts.show()

print("Data Type Checks:")
df_check.show()

In [0]:
# Remove Nulls from Critical Fields
clean_df = rawcustomers_df.filter(
    F.col("customer_id").isNotNull() &
    F.col("name").isNotNull() &
    F.col("email").isNotNull()
)

# Validate Email Format
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

clean_df = clean_df.filter(F.col("email").rlike(email_regex))


valid_statuses = ["active", "inactive"]
clean_df = clean_df.filter( F.col("status").isin(valid_statuses))


# Filter out valid records - Fixed boolean logic
df_valid_booking = clean_df.filter(
        (F.col("customer_id").isNotNull()) & 
        (F.col("name").isNotNull()) 
    )
    
    # Capture invalid records for error handling - Fixed boolean logic
df_invalid_booking = clean_df.filter(
        (F.col("customer_id").isNull()) | 
        (F.col("name").isNull()) 
    )
    
valid_records = df_valid_booking.count()
invalid_records = df_invalid_booking.count()

print(f"Valid records: {valid_records}")
print(f"Invalid records: {invalid_records}")

clean_df.show()


In [0]:
# Write valid data to staging table
try:
    # Create or overwrite staging table
    df_valid_booking.write.format("delta").mode("overwrite").saveAsTable(stage_table)
    print(f"Successfullyloaded {valid_records} valid booking to staging table")
    
    # Write invalid records to error table for investigation
    if invalid_records > 0:
        df_invalid_booking.write.withColumn("error_reason", F.lit("Data quality validation failed")) \
                           .withColumn("error_timestamp", F.current_timestamp()) \
                           .write.format("delta").mode("append").saveAsTable(error_table)
        print(f"Logged {invalid_records} invalid records to error table")
    
except Exception as e:
    print(f"Error writing to staging table: {str(e)}")
    raise

In [0]:
# Archive processed files
try:
    # List all files in the source directory
    files = dbutils.fs.ls(source_dir)
    
    archived_count = 0
    for file in files:
        if file.name.endswith('.json'):
            src_path = file.path
            archive_path = archive_dir + file.name
            
            # Move the file to archive
            dbutils.fs.mv(src_path, archive_path)
            archived_count += 1
            print(f"Archived: {file.name}")
    
    print(f"Successfully archived {archived_count} files")
    
except Exception as e:
    print(f"Error archiving files: {str(e)}")
    raise